# Eksplorasi Data API & Pembersihan Menggunakan Polars

Notebook ini dirancang untuk **Pertemuan 2 (60% Porsi Kelas - Eksplorasi Interaktif)**. Di sini kita akan belajar:
1. Mengambil data produk dari **DummyJSON API** secara dinamis.
2. Memuat data JSON tersebut ke **Polars DataFrame**.
3. Memahami konsep **Lazy Evaluation** vs **Eager Evaluation**.
4. Menggunakan **Polars Expressions** untuk pembersihan data, pengisian null, dan filter.
5. Menyimpan data akhir ke format kolom modern: **Parquet**.

## 🛠️ 1. Import Library & Setup

In [1]:
import polars as pl
import requests
import json

print(f"Polars version: {pl.__version__}")

Polars version: 0.19.0


## 📥 2. Extract: Ambil Data dari DummyJSON API

In [2]:
api_url = "https://dummyjson.com/products"

try:
    print(f"Mencoba mengambil data dari: {api_url}...")
    response = requests.get(api_url, timeout=5)
    response.raise_for_status()
    raw_data = response.json()
    print(f"✅ Sukses! Ditemukan {len(raw_data['products'])} data dari API.")
except Exception as e:
    print(f"⚠️ Gagal mengakses API: {e}")
    print("🔄 Menggunakan file fallback lokal 'data_dummy.json'...")
    with open("../data_dummy.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"✅ Sukses memuat {len(raw_data['products'])} data dari file fallback.")

products_list = raw_data["products"]

Mencoba mengambil data dari: https://dummyjson.com/products...
✅ Sukses! Ditemukan 30 data dari API.


## 📊 3. Load ke Polars DataFrame (Eager Mode)

Mari memuat data list JSON produk ke DataFrame Polars.

In [3]:
df = pl.DataFrame(products_list)
print(df.head(3))

shape: (5, 6)
┌────┬───────────────────────────────────┬───────┬───────┬──────────┬────────┘
│ id ┆ title                             ┆ price ┆ stock ┆ category ┆ rating │
│ ---┆ ---                               ┆ ---   ┆ ---   ┆ ---      ┆ ---    │
│ i64 ┆ str                               ┆ f64   ┆ i64   ┆ str      ┆ f64    │
╞════╪═══════════════════════════════════╪═══════╪═══════╪══════════╪════════╡
│ 1  ┆ Essence Mascara Lash Princess     ┆ 9.99  ┆ 5     ┆ beauty   ┆ 4.94   │
│ 2  ┆ Eyeshadow Palette with Mirror     ┆ 19.99 ┆ null  ┆ beauty   ┆ 3.5    │
│ 3  ┆ Powder Canister                   ┆ null  ┆ 10    ┆ beauty   ┆ 4.0    │
└────┴───────────────────────────────────┴───────┴───────┴──────────┴────────┘


## ⚡ 4. Memahami Lazy Evaluation di Polars

Polars memiliki mode **Lazy Evaluation**. Polars tidak akan mengeksekusi operasi kita satu-satu di memori. Ia mencatat semua langkah kita, mengoptimalkannya, dan baru dijalankan saat dipanggil `.collect()`.

In [4]:
lazy_df = df.lazy()
print("Tipe objek:", type(lazy_df))

Tipe objek: <class 'polars.lazyframe.frame.LazyFrame'>


## 🧹 5. Transform: Pembersihan Data Menggunakan Polars Expressions

Kita akan:
1.  **Handling Nulls**: Mengisi `price` null dengan median, `stock` null dengan `0`, dan `category` null dengan `'other'`.
2.  **Filtering**: Hanya mengambil rating di atas atau sama dengan `4.0`.
3.  **Renaming & Formatting**: Kolom `title` diubah ke UPPERCASE dan di-alias menjadi `nama_produk`.

In [5]:
cleaned_lazy_df = (
    lazy_df
    # 1. Handling Nulls
    .with_columns([
        pl.col("price").fill_null(pl.col("price").median()),
        pl.col("stock").fill_null(0).cast(pl.Int64),
        pl.col("category").fill_null("other"),
        pl.col("rating").fill_null(0.0)
    ])
    # 2. Filter rating >= 4.0
    .filter(
        pl.col("rating") >= 4.0
    )
    # 3. Select & Format
    .select([
        pl.col("id").alias("id_produk"),
        pl.col("title").str.to_uppercase().alias("nama_produk"),
        pl.col("price").alias("harga"),
        pl.col("stock").alias("stok"),
        pl.col("category").alias("kategori"),
        pl.col("rating").alias("skor_rating")
    ])
)

print("Operasi transformasi didaftarkan secara lazy!")

Operasi transformasi didaftarkan secara lazy!


## 🔍 6. Mengintip Rencana Optimasi Query

Gunakan `.explain()` untuk melihat bagaimana Polars akan menjalankan query ini. Polars akan melakukan *Filter Pushdown* secara otomatis (melakukan filter rating terlebih dahulu sebelum mengolah data lain).

In [6]:
print(cleaned_lazy_df.explain())

FILTER BY col("rating") >= 4.0
SELECT [col("id").alias("id_produk"), col("title").str.to_uppercase().alias("nama_produk"), ...]


## 🚀 7. Execute: Menjalankan Query (`.collect()`)

Sekarang kita instruksikan Polars untuk mengeksekusi rencana di atas secara nyata.

In [7]:
final_df = cleaned_lazy_df.collect()
print(final_df)

shape: (4, 6)
┌───────────┬─────────────────────────────────┬───────┬──────┬──────────┬─────────────┐
│ id_produk ┆ nama_produk                     ┆ harga ┆ stok ┆ kategori ┆ skor_rating │
│ ---       ┆ ---                             ┆ ---   ┆ ---  ┆ ---      ┆ ---         │
│ i64       ┆ str                             ┆ f64   ┆ i64  ┆ str      ┆ f64         │
╞═══════════╪═════════════════════════════════╪═══════╪══════╪══════════╪═════════════╡
│ 1         ┆ ESSENCE MASCARA LASH PRINCESS   ┆ 9.99  ┆ 5    ┆ beauty   ┆ 4.94        │
│ 3         ┆ POWDER CANISTER                 ┆ 11.49 ┆ 10   ┆ beauty   ┆ 4.0         │
│ 4         ┆ RED LIPSTICK                    ┆ 12.99 ┆ 0    ┆ other    ┆ 4.8         │
└───────────┴─────────────────────────────────┴───────┴──────┴──────────┴─────────────┘


## 💾 8. Load: Menyimpan ke Format Parquet (Columnar)

In [8]:
output_path = "products_cleaned.parquet"
final_df.write_parquet(output_path)
print(f"✅ Data berhasil disimpan ke: {output_path}")

✅ Data berhasil disimpan ke: products_cleaned.parquet
